# Cosmic Engine - GPU-Accelerated AI Rendering

This notebook provides GPU-accelerated AI rendering for the Cosmic Engine project.

## Workflow:
1. **Local Machine (Phases 1-2)**: Generate L-system geometry (.obj files)
2. **Google Colab (Phase 3)**: AI-synthesize photorealistic images with SDXL

---

### Prerequisites
- Enable GPU: Runtime > Change runtime type > GPU
- Upload .obj files from local machine to Colab

---

In [ ]:
# Check GPU availability
!nvidia-smi

## Step 1: Setup Project

Clone the project or upload your local files.

In [ ]:
# Option A: Clone from repository (if you have one)
# !git clone YOUR_REPO_URL cosmic_engine
# %cd cosmic_engine

# Option B: Use Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Navigate to your project folder in Drive
# %cd /content/drive/MyDrive/cosmic_engine

# Option C: Upload files directly
# from google.colab import files
# uploaded = files.upload()  # Upload your .obj files

## Step 2: Install Dependencies with CUDA Support

In [ ]:
# Install PyTorch with CUDA 11.8 (adjust for Colab's CUDA version)
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu118

# Install other dependencies
!pip install numpy numba pandas matplotlib Pillow opencv-python

# Install AI dependencies for Phase 3
!pip install diffusers transformers accelerate xformers

## Step 3: Verify CUDA Setup

In [ ]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## Step 4: Upload or Reference .OBJ Files

Upload your locally-generated geometry files.

In [ ]:
# Create directories for input/output
!mkdir -p input_geometries
!mkdir -p output_gallery

# Upload .obj files
from google.colab import files
uploaded = files.upload()

# Move uploaded files to input directory
!mv *.obj input_geometries/ 2>/dev/null || true

# List uploaded files
!ls -lh input_geometries/

## Step 5: Check Factory Capabilities

In [ ]:
# List available styles
!python factory.py --list-styles

# Check system info
!python factory.py --system-info

## Step 6: Generate Maps (Fast, No GPU Required)

First, generate analysis maps from your geometry.

In [ ]:
# Choose your .obj file
INPUT_FILE = "input_geometries/alien_test.obj"
OUTPUT_DIR = "output_gallery/alien_maps"

!python factory.py --input {INPUT_FILE} --maps-only --output {OUTPUT_DIR}

# Display generated maps
from IPython.display import Image, display
import os

for img_file in sorted(os.listdir(OUTPUT_DIR)):
    if img_file.endswith('.png'):
        print(f"\n{img_file}:")
        display(Image(os.path.join(OUTPUT_DIR, img_file), width=400))

## Step 7: AI Image Synthesis (GPU Required)

Now use Stable Diffusion XL to synthesize photorealistic images.

### 7a. Alien Structure (Negative Phototropism, High Pipe Exponent)

In [ ]:
INPUT_FILE = "input_geometries/alien_test.obj"
OUTPUT_DIR = "output_gallery/alien_renders"
STYLE = "alien"

!python factory.py \
    --input {INPUT_FILE} \
    --style {STYLE} \
    --output {OUTPUT_DIR} \
    --resolution 1024 \
    --samples 4

# Display results
for img_file in sorted(os.listdir(OUTPUT_DIR)):
    if img_file.endswith('.png') and 'render' in img_file:
        print(f"\n{img_file}:")
        display(Image(os.path.join(OUTPUT_DIR, img_file), width=600))

### 7b. Prehistoric Mega-Flora (Low Pipe Exponent, Thick Trunks)

In [ ]:
INPUT_FILE = "input_geometries/prehistoric_test.obj"
OUTPUT_DIR = "output_gallery/prehistoric_renders"
STYLE = "prehistoric"

!python factory.py \
    --input {INPUT_FILE} \
    --style {STYLE} \
    --output {OUTPUT_DIR} \
    --resolution 1024 \
    --samples 4

# Display results
for img_file in sorted(os.listdir(OUTPUT_DIR)):
    if img_file.endswith('.png') and 'render' in img_file:
        print(f"\n{img_file}:")
        display(Image(os.path.join(OUTPUT_DIR, img_file), width=600))

### 7c. Custom Style with Manual Prompt

In [ ]:
INPUT_FILE = "input_geometries/your_structure.obj"
OUTPUT_DIR = "output_gallery/custom_renders"
CUSTOM_PROMPT = "bioluminescent crystalline trees in deep ocean, volumetric lighting, cinematic"

!python factory.py \
    --input {INPUT_FILE} \
    --prompt "{CUSTOM_PROMPT}" \
    --output {OUTPUT_DIR} \
    --resolution 1024 \
    --samples 2

# Display results
for img_file in sorted(os.listdir(OUTPUT_DIR)):
    if img_file.endswith('.png') and 'render' in img_file:
        print(f"\n{img_file}:")
        display(Image(os.path.join(OUTPUT_DIR, img_file), width=600))

## Step 8: Batch Processing Multiple Structures

In [ ]:
import os

# Process all .obj files in input directory
input_dir = "input_geometries"
styles = ["alien", "prehistoric", "bioluminescent"]

for obj_file in os.listdir(input_dir):
    if obj_file.endswith('.obj'):
        base_name = obj_file.replace('.obj', '')
        
        for style in styles:
            input_path = os.path.join(input_dir, obj_file)
            output_dir = f"output_gallery/{base_name}_{style}"
            
            print(f"\n{'='*60}")
            print(f"Processing: {obj_file} with style: {style}")
            print(f"{'='*60}")
            
            !python factory.py \
                --input {input_path} \
                --style {style} \
                --output {output_dir} \
                --resolution 1024 \
                --samples 2

print("\n✓ Batch processing complete!")

## Step 9: Download Results

In [ ]:
# Zip all output
!zip -r cosmic_engine_renders.zip output_gallery/

# Download
from google.colab import files
files.download('cosmic_engine_renders.zip')

# Or save to Google Drive
# !cp -r output_gallery /content/drive/MyDrive/cosmic_engine_outputs/

---

## Advanced: Memory Optimization

If you encounter OOM (Out of Memory) errors:

In [ ]:
# 1. Lower resolution
!python factory.py --input input.obj --style alien --resolution 768

# 2. Use FP32 instead of FP16 (paradoxically helps on some GPUs)
!python factory.py --input input.obj --style alien --no-fp16

# 3. Clear GPU memory between runs
import torch
torch.cuda.empty_cache()

# 4. Check memory usage
!nvidia-smi

---

## Tips for Novel Trilogy Development

### Alien Structures
- Use **negative phototropism** (grows away from light)
- High **pipe_exponent** (2.5-3.0) for spindly, top-heavy forms
- Asymmetric branching angles
- Try styles: `alien`, `bioluminescent`, `crystalline`

### Prehistoric Mega-Flora
- Low **pipe_exponent** (1.2-1.8) for thick trunks
- Model after Lepidodendron, Sigillaria
- Sparse branching patterns
- Try styles: `prehistoric`, `carboniferous`, `swamp`

### Workflow Recommendations
1. Generate 50-100 candidate structures locally (fast)
2. Filter by geometry metrics (trunk diameter, height, branch count)
3. Upload best 10-20 to Colab for AI rendering
4. Iterate on prompts to achieve "uncanny" aesthetic

---

## Resources

- **Project Docs**: `README.md`, `BIOSIM_README.md`, `FACTORY_README.md`
- **L-System Examples**: `examples/` directory
- **Hugging Face SDXL**: https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0

---

*Generated for the Cosmic Engine project - Phases 1-4*